# 01 - Data understanding: NUCLE

Use the shared parser and CSV schema for notebook exploration and the report. Titles occupy text block 0 when present. Internal whitespace is preserved. Run cells from top to bottom. Requires pandas.

If a cell fails, fix the error, restart the kernel, and run all cells from the top. Paths come from the shared parser and currently use `data_set/`.


In [1]:
from pathlib import Path
from collections import Counter
import sys

# Jupyter has no __file__; locate the repository from the kernel directory.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/data/parse_nucle.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook with the kernel working directory inside the project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.nucle_sgml import load_nucle
from src.data.parse_nucle import (
    documents_to_dataframe, TARGET_LABELS, DEFAULT_INPUT, DEFAULT_OUTPUT,
)
from src.data.report_nucle import build_report

DATA_PATH = DEFAULT_INPUT
OUTPUT_PATH = DEFAULT_OUTPUT
docs = load_nucle(DATA_PATH)
if not docs:
    raise ValueError("No documents found")
df = documents_to_dataframe(docs)
print(f"Documents: {len(docs):,}; annotations: {len(df):,}")
print("First text block:", docs[0].paragraphs[0] if docs[0].paragraphs else "<missing>")
print(df[["essay_id", "error_span", "error_type", "correction", "valid_offset"]].head(3).to_string(index=False))


Documents: 1,397; annotations: 44,912
First text block: CREATING A HABITABLE ENVIRONMENT
essay_id error_span error_type correction  valid_offset
     829     caused      Vform      cause          True
     829        are        SVA         is          True
     829     spaces         Nn      space          True


## Data quality

Invalid endpoints remain flagged. Cross-paragraph annotations are validated but excluded from single-paragraph span diagnostics. Insertions below mean valid equal start/end offsets.


In [2]:
ids = [doc.essay_id for doc in docs]
quality = {
    "documents": len(docs),
    "missing_essay_ids": sum(not value for value in ids),
    "duplicate_essay_ids": sum(count - 1 for key, count in Counter(ids).items() if key),
    "documents_without_text_blocks": sum(not doc.paragraphs for doc in docs),
    "documents_without_annotations": sum(not doc.mistakes for doc in docs),
    "annotations": len(df),
    "invalid_spans": int((~df.valid_offset).sum()),
    "missing_error_types": int(df.error_type.eq("").sum()),
    "missing_correction_tags": int(df.missing_correction.sum()),
    "empty_corrections": int(df.correction.eq("").sum()),
    "insertions": int(df.is_insertion.sum()),
    "cross_paragraph_annotations": int(df.cross_paragraph.sum()),
    "duplicate_annotation_records": int(df.duplicated(subset=[
        "essay_id", "start_par", "start_off", "end_par", "end_off",
        "error_type", "correction"]).sum()),
}
for key, value in quality.items():
    print(f"{key}: {value:,}")
print("Invalid annotation preview:")
print(df.loc[~df.valid_offset].head(3).to_string(index=False))


documents: 1,397
missing_essay_ids: 0
duplicate_essay_ids: 0
documents_without_text_blocks: 0
documents_without_annotations: 0
annotations: 44,912
invalid_spans: 0
missing_error_types: 0
missing_correction_tags: 0
empty_corrections: 10,004
insertions: 0
cross_paragraph_annotations: 4
duplicate_annotation_records: 0
Invalid annotation preview:
Empty DataFrame
Columns: [annotation_id, essay_index, essay_id, mistake_index, start_par, start_off, end_par, end_off, paragraph, error_span, error_type, correction, comment, missing_correction, edit_type, is_target_label, is_insertion, cross_paragraph, valid_offset]
Index: []


## Error distribution and examples

Counts are annotations, not independent training examples. Target counts include rows flagged for review.


In [3]:
print(df.error_type.value_counts().to_string())
target_df = df[df.is_target_label]
coverage = len(target_df) / len(df) if len(df) else 0
print(f"Target coverage: {len(target_df):,}/{len(df):,} ({coverage:.2%})")
for label in sorted(TARGET_LABELS):
    print("\n" + label)
    examples = target_df[(target_df.error_type == label) & target_df.valid_offset & ~target_df.cross_paragraph]
    print(examples[["error_span", "correction"]].head(3).to_string(index=False))


error_type
ArtOrDet    6640
Wci         5305
Rloc-       4703
Nn          3768
Vt          3204
Mec         3145
Prep        2413
Wform       2161
SVA         1524
Others      1467
Vform       1443
Trans       1377
Um          1164
Pref         927
Srun         873
WOinc        698
Cit          658
Wtone        593
Spar         519
Vm           431
V0           414
Ssub         362
WOadv        347
Sfrag        250
Npos         239
Pform        186
Smod          51
Wa            50
Target coverage: 15,788/44,912 (35.15%)

ArtOrDet
error_span correction
       the      their
       the           
         a           

Nn
error_span correction
    spaces      space
   citizen   citizens
   problem   problems

Prep
error_span correction
        in         of
        of        for
        in           

SVA
error_span correction
       are         is
       use       uses
      have        has

Vform
error_span correction
    caused      cause
    obtain   obtained
        is        are


## Shared data understanding report

This is the same report generator used by `python -m src.data.report_nucle`. Span edit categories do not necessarily identify linguistic missing-word errors.


In [4]:
print(build_report(df))


# NUCLE Data Understanding Report

## 1. Dataset overview

- Total annotation rows: **44,912**
- Unique essays represented in annotations: **1,397**
- Unique error types: **28**
- Target-label rows: **15,788**
- Invalid offsets: **0**
- Cross-paragraph annotations: **4**

## 2. Five target classes

| error_type | count | percentage | essays_with_class |
| --- | --- | --- | --- |
| ArtOrDet | 6640 | 42.06 | 1178 |
| Prep | 2413 | 15.28 | 954 |
| Nn | 3768 | 23.87 | 1027 |
| Vform | 1443 | 9.14 | 690 |
| SVA | 1524 | 9.65 | 746 |

## 3. Edit-type distribution inside the five-class subset

| edit_type | count |
| --- | --- |
| replacement | 12821 |
| deletion | 2967 |

## 4. Target errors per essay

- Essays containing at least one target error: **1,355**
- Mean errors per essay: **11.65**
- Median errors per essay: **9.00**
- Minimum: **1**
- Maximum: **62**

## 5. Span-length diagnostics

Character-length statistics for valid, single-paragraph target spans only.
Excluded target rows: **

## Optional export

Export uses the shared schema expected by the report script. Split by essay ID during later preparation and fit learned preprocessing on training data only.


In [5]:
EXPORT_CSV = False
if EXPORT_CSV:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(df):,} annotations to {OUTPUT_PATH}")
else:
    print("Export disabled. Set EXPORT_CSV = True to save the shared annotation table.")


Export disabled. Set EXPORT_CSV = True to save the shared annotation table.
